<a id="top"></a>
# Marketing bancario: preprocessing con Pipeline di scikit-learn

Questo notebook usa il dataset **Bank Marketing** del repository UCI per costruire un esempio minimale di preprocessing con gli strumenti di scikit-learn. L'obiettivo non è ottimizzare il modello, ma mostrare come trasformare un sottoinsieme di variabili in una matrice di feature pronta per l'addestramento.

<a id="indice"></a>
## Indice

- [Obiettivo e quadro generale](#obiettivo)
- [Caricamento del dataset Bank Marketing](#caricamento)
- [Scelta di un sottoinsieme di variabili](#selezione-sottoinsieme)
- [Configurare scikit-learn per lavorare con DataFrame](#configurazione-scikit)
- [Encoding delle variabili binarie](#binarie)
- [Encoding delle variabili categoriali nominali](#nominali)
- [Encoding delle variabili categoriali ordinali](#ordinali)
- [Trattamento delle variabili numeriche](#numeriche)
- [Costruire un preprocessore unico con ColumnTransformer](#pipeline-preprocessore)
- [Aggiungere un modello di classificazione alla pipeline](#pipeline-modello)
- [Perché salvare pipeline + modello](#sintesi-pipeline)


<a id="obiettivo"></a>
## Obiettivo e quadro generale

Il dataset di marketing bancario contiene variabili eterogenee: numeriche, binarie, categoriali nominali e categoriali ordinali. In questo notebook si lavora su un sottoinsieme ridotto di colonne, scelto per coprire tutti questi casi tipici.

Il percorso segue tre idee semplici:

- definire esplicitamente quali variabili appartengono a ciascun tipo;
- applicare a ogni gruppo un trasformatore di scikit-learn adatto (`OrdinalEncoder`, `OneHotEncoder`, `StandardScaler`, …);
- raccogliere tutto in una singola `Pipeline` che combina preprocessing e modello.

Il risultato finale è un oggetto unico (pipeline + modello) che può essere riutilizzato senza dover riscrivere, a mano, l'intera sequenza di trasformazioni.


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Nota importante**

Nel dataset originale la colonna `duration` (secondi dell'ultima chiamata) è fortemente predittiva del target, ma non è disponibile nel momento in cui bisogna decidere **chi chiamare**.  
Per evitare leakage, in questo notebook `duration` viene esclusa dal sottoinsieme di variabili usate per il modello.

</div>


<a id="caricamento"></a>
## Caricamento del dataset Bank Marketing

Il dataset viene caricato dal repository UCI tramite la funzione `fetch_ucirepo`.  
Si costruisce un unico `DataFrame` con feature (`X`) e target (`y`), mantenendo i nomi originali delle colonne.


In [5]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

# Caricamento del dataset Bank Marketing (id=222 nel repository UCI)
bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

# Costruiamo un unico DataFrame con feature e target
df = X.copy()
if isinstance(y, pd.DataFrame):
    if y.shape[1] == 1:
        df["y"] = y.iloc[:, 0]
    else:
        for col in y.columns:
            df[col] = y[col]
else:
    df["y"] = y

print("Dimensione feature:", X.shape)
print("Dimensione target:", y.shape)
df.head()

Dimensione feature: (45211, 16)
Dimensione target: (45211, 1)


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


<a id="selezione-sottoinsieme"></a>
## Scelta di un sottoinsieme di variabili

Per rendere più chiaro il lavoro di preprocessing si lavora su un sottoinsieme di colonne che copre i casi più frequenti:

- variabili binarie (`yes` / `no`);
- variabili categoriali nominali (categorie senza ordine intrinseco);
- variabili categoriali ordinali (categorie con un ordine naturale);
- variabili numeriche.

Nel dataset Bank Marketing queste scelte sono:

| Tipo                       | Colonne scelte                                      |
|----------------------------|-----------------------------------------------------|
| Binarie                    | `default`, `housing`                                |
| Categoriali nominali       | `job`, `marital`, `poutcome`                        |
| Categoriali ordinali       | `education`, `month`                                |
| Numeriche                  | `age`, `balance`, `pdays`, `previous`               |

La variabile `poutcome` descrive l'esito della precedente campagna di marketing e viene trattata come categoriale nominale.  
Il target di classificazione resta `y` (sottoscrizione del deposito: `"yes"` / `"no"`).


In [6]:
import pandas as pd

target_col = "y"

binary_cols = ["default", "housing"]
nominal_cols = ["job", "marital", "poutcome"]
ordinal_cols = ["education", "month"]
numeric_cols = ["age", "balance", "pdays", "previous"]

feature_cols = binary_cols + nominal_cols + ordinal_cols + numeric_cols

df_model = df[feature_cols + [target_col]].copy()
df_model.head()

,default,housing,job,marital,poutcome,education,month,age,balance,pdays,previous,y
0,no,yes,management,married,NaN,tertiary,may,58,2143,-1,0,no
1,no,yes,technician,single,NaN,secondary,may,44,29,-1,0,no
2,no,yes,entrepreneur,married,NaN,secondary,may,33,2,-1,0,no
3,no,yes,blue-collar,married,NaN,NaN,may,47,1506,-1,0,no
4,no,no,NaN,single,NaN,NaN,may,33,1,-1,0,no


<a id="configurazione-scikit"></a>
## Configurare scikit-learn per lavorare con DataFrame

Per leggere con chiarezza l'effetto di ogni step di preprocessing conviene chiedere a scikit-learn di restituire `DataFrame` invece di array NumPy anonimi.

Inoltre si prepara uno split train/test: gli oggetti di scikit-learn imparano dai dati di train tramite `fit(…)` e applicano la trasformazione (o la predizione) a nuovi dati tramite `transform(…)` o `predict(…)`.


In [7]:
from sklearn import set_config
from sklearn.model_selection import train_test_split

set_config(transform_output="pandas")

X = df_model[feature_cols]
y = df_model[target_col].map({"no": 0, "yes": 1}).astype("int8")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.head()

,default,housing,job,marital,poutcome,education,month,age,balance,pdays,previous
24001,no,no,technician,divorced,NaN,secondary,aug,36,861,-1,0
43409,no,no,student,single,failure,secondary,apr,24,4126,185,7
20669,no,yes,technician,single,NaN,secondary,aug,44,244,-1,0
18810,no,no,unemployed,married,NaN,secondary,jul,48,0,-1,0
23130,no,no,technician,married,NaN,secondary,aug,38,257,-1,0


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Attenzione**

Tutti i trasformatori di scikit-learn (encoder, scaler, ecc.) devono essere **fittati solo sui dati di training**.  
Su validation e test si usano esclusivamente i metodi `transform(…)` e `predict(…)` degli oggetti già fittati.  
Mescolare i ruoli porta a leakage e a stime troppo ottimistiche delle prestazioni.

</div>


<a id="binarie"></a>
## Encoding delle variabili binarie

Le variabili `default` e `housing` sono codificate come stringhe `"yes"` / `"no"`.  
Per i modelli numerici è comodo rappresentarle come 0/1, preservando il significato di ordine naturale: 0 = assenza della caratteristica, 1 = presenza.

Si usa `OrdinalEncoder` di scikit-learn, specificando esplicitamente l'ordine delle categorie per ciascuna colonna.


In [8]:
from sklearn.preprocessing import OrdinalEncoder

binary_encoder = OrdinalEncoder(
    categories=[["no", "yes"], ["no", "yes"]],
    dtype=float,
)

binary_encoder.fit(X_train[binary_cols])

X_train_bin = binary_encoder.transform(X_train[binary_cols])
X_train_bin.head()

,default,housing
24001,0.0,0.0
43409,0.0,0.0
20669,0.0,1.0
18810,0.0,0.0
23130,0.0,0.0


<a id="nominali"></a>
## Encoding delle variabili categoriali nominali

Le variabili `job`, `marital` e `poutcome` descrivono categorie senza un ordine naturale.  
In questi casi si usa un encoding one-hot, che produce una colonna per ogni categoria possibile e assegna 0/1 a seconda dell'appartenenza.

Per gestire categorie mai viste in training si imposta `handle_unknown="ignore"`.


In [17]:
from sklearn.preprocessing import OneHotEncoder

nominal_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

nominal_encoder.fit(X_train[nominal_cols])

X_train_nom = nominal_encoder.transform(X_train[nominal_cols])
X_train_nom.head()

,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_nan,marital_divorced,marital_married,marital_single,poutcome_failure,poutcome_other,poutcome_success,poutcome_nan
24001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
43409,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
20669,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
18810,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
23130,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [20]:
# Apply the fitted binary encoder to the held-out set without refitting it
X_test_bin = binary_encoder.transform(X_test[binary_cols])

print("Encoded categories:", binary_encoder.categories_)
X_test_bin.head()

Encoded categories: [array(['no', 'yes'], dtype=object), array(['no', 'yes'], dtype=object)]


,default,housing
1392,0.0,1.0
7518,0.0,1.0
12007,0.0,1.0
5536,0.0,1.0
29816,0.0,1.0


<a id="ordinali"></a>
## Encoding delle variabili categoriali ordinali

Le variabili `education` e `month` hanno un ordine naturale:

- i livelli di istruzione vanno da `illiterate` a `university.degree`;
- i mesi vanno da gennaio a dicembre.

Per queste colonne ha senso usare un encoding ordinal che rispetti l'ordine logico delle categorie.


In [14]:
education_order = ["primary", "secondary", "tertiary"]

month_order = [
    "jan", "feb", "mar", "apr", "may", "jun",
    "jul", "aug", "sep", "oct", "nov", "dec",
]

from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder(
    categories=[education_order, month_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=float,
)

ordinal_encoder.fit(X_train[ordinal_cols])

X_train_ord = ordinal_encoder.transform(X_train[ordinal_cols])
X_train_ord.head()

,education,month
24001,1.0,7.0
43409,1.0,3.0
20669,1.0,7.0
18810,1.0,6.0
23130,1.0,7.0


<a id="numeriche"></a>
## Trattamento delle variabili numeriche

Le variabili numeriche scelte (`age`, `balance`, `pdays`, `previous`) hanno scale molto diverse tra loro.  
Uno step tipico di preprocessing è la standardizzazione: si sottrae la media e si divide per la deviazione standard calcolate sui dati di training.

Per farlo si usa `StandardScaler` di scikit-learn.


In [12]:
from sklearn.preprocessing import StandardScaler

numeric_scaler = StandardScaler()

numeric_scaler.fit(X_train[numeric_cols])

X_train_num = numeric_scaler.transform(X_train[numeric_cols])
X_train_num.head()

,age,balance,pdays,previous
24001,-0.460434,-0.164410,-0.410910,-0.241509
43409,-1.589641,0.899627,1.446096,2.664584
20669,0.292371,-0.365486,-0.410910,-0.241509
18810,0.668773,-0.445003,-0.410910,-0.241509
23130,-0.272233,-0.361249,-0.410910,-0.241509


<a id="pipeline-preprocessore"></a>
## Costruire un preprocessore unico con ColumnTransformer

Finora gli step sono stati applicati separatamente, ma in pratica è comodo avere un solo oggetto che conosce:

- quali colonne prendere;
- quale trasformatore applicare a ciascun gruppo.

`ColumnTransformer` permette di combinare più trasformatori in parallelo e restituire una matrice di feature unica.


In [21]:
from sklearn.compose import ColumnTransformer

preprocess = ColumnTransformer(
    transformers=[
        ("bin", binary_encoder, binary_cols),
        ("ord", ordinal_encoder, ordinal_cols),
        ("nom", nominal_encoder, nominal_cols),
        ("num", numeric_scaler, numeric_cols),
    ]
)

X_train_pre = preprocess.fit_transform(X_train)
X_train_pre.head()

,bin__default,bin__housing,ord__education,ord__month,nom__job_admin.,nom__job_blue-collar,nom__job_entrepreneur,nom__job_housemaid,nom__job_management,nom__job_retired,...,nom__marital_married,nom__marital_single,nom__poutcome_failure,nom__poutcome_other,nom__poutcome_success,nom__poutcome_nan,num__age,num__balance,num__pdays,num__previous
24001,0.0,0.0,1.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,-0.460434,-0.164410,-0.410910,-0.241509
43409,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,-1.589641,0.899627,1.446096,2.664584
20669,0.0,1.0,1.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.292371,-0.365486,-0.410910,-0.241509
18810,0.0,0.0,1.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.668773,-0.445003,-0.410910,-0.241509
23130,0.0,0.0,1.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,-0.272233,-0.361249,-0.410910,-0.241509


<a id="pipeline-modello"></a>
## Aggiungere un modello di classificazione alla pipeline

Per chiudere il cerchio si aggiunge un modello di classificazione in fondo al preprocessore.  
Si usa una regressione logistica, sufficiente per illustrare il flusso `fit` → `transform` → `predict`.


In [22]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

clf.fit(X_train, y_train)

y_proba = clf.predict_proba(X_test)[:, 1]
y_pred = clf.predict(X_test)

auc = roc_auc_score(y_test, y_proba)
print(f"ROC AUC (test): {auc:.3f}")
print()
print(classification_report(y_test, y_pred))

ROC AUC (test): 0.714

              precision    recall  f1-score   support

           0       0.90      0.99      0.94      7985
           1       0.68      0.17      0.27      1058

    accuracy                           0.89      9043
   macro avg       0.79      0.58      0.61      9043
weighted avg       0.87      0.89      0.86      9043



In [23]:
clf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['default','housing','job',...,'balance','pdays','previous']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('bin', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining

In [24]:
clf.named_steps['model'].__dict__




{'penalty': 'deprecated',
 'C': 1.0,
 'l1_ratio': 0.0,
 'dual': False,
 'tol': 0.0001,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'class_weight': None,
 'random_state': None,
 'solver': 'lbfgs',
 'max_iter': 1000,
 'verbose': 0,
 'warm_start': False,
 'n_jobs': None,
 'feature_names_in_': array(['bin__default', 'bin__housing', 'ord__education', 'ord__month',
        'nom__job_admin.', 'nom__job_blue-collar', 'nom__job_entrepreneur',
        'nom__job_housemaid', 'nom__job_management', 'nom__job_retired',
        'nom__job_self-employed', 'nom__job_services', 'nom__job_student',
        'nom__job_technician', 'nom__job_unemployed', 'nom__job_nan',
        'nom__marital_divorced', 'nom__marital_married',
        'nom__marital_single', 'nom__poutcome_failure',
        'nom__poutcome_other', 'nom__poutcome_success',
        'nom__poutcome_nan', 'num__age', 'num__balance', 'num__pdays',
        'num__previous'], dtype=object),
 'n_features_in_': 27,
 'classes_': array([0, 1], dtype=i

<a id="sintesi-pipeline"></a>
## Perché salvare pipeline + modello

La pipeline `clf` contiene sia il modello sia tutte le trasformazioni applicate alle colonne originali:

- mapping delle variabili binarie;
- encoding one-hot delle categoriali nominali;
- encoding ordinal delle categoriali ordinali;
- standardizzazione delle variabili numeriche.

Applicare il modello a nuovi dati grezzi significa passare un `DataFrame` con le stesse colonne di `X_train`:

```python
y_new_proba = clf.predict_proba(df_nuovi_clienti[feature_cols])[:, 1]
```

La logica di preprocessing è riutilizzata automaticamente e resta coerente tra addestramento, validazione e messa in produzione.

Senza una pipeline unica sarebbe necessario ricordare, copiare e tenere allineati tutti i passaggi di trasformazione.  
Con un solo oggetto serializzabile (ad esempio tramite MLflow o `joblib`) si porta con sé sia il modello sia la ricetta completa per preparare i dati.
